# Session 1: NumPy Fundamentals (2 hours)

**Goal:** Build the mental model: *read an ML-style task → map it to array shapes and operations → implement in NumPy.* We start with real mini-tasks (cases), not a long list of basics.

| Section | Time | Focus |
|---------|------|-------|
| Setup | 10 min | Import & verify environment |
| Anchor case 1 | 40 min | Weighted sum & batch prediction |
| Anchor case 2 | 30 min | Normalise a vector to 0–1 |
| Anchor case 3 | 35 min | Boolean indexing (extract with mask) |
| Anchor case 4 | 35 min | View vs copy — avoid the bug |
| Reference | 50 min | NumPy essentials drill (if you need more practice) |

## Setup (10 min)

In [1]:
import numpy as np

print(f"NumPy version: {np.__version__}")
print("Setup complete!")

NumPy version: 1.26.4
Setup complete!


---
## Anchor case 1: Weighted sum and batch prediction [EASY]

In ML we often compute **weighted sums**: one number per example (e.g. a score or prediction). Here we do it for **one** example, then for a **batch** of examples (matrix × vector).

### Concept: Arrays and shape

An **array** is a grid of numbers. **Shape** is the size along each dimension: e.g. `(5,)` = 1D vector of length 5, `(3, 4)` = 3 rows, 4 columns. Use `arr.shape` to inspect. Create from a list with `np.array([...])`.

### Worked example: One neuron, then a batch

**Task:** Given scores (features) and weights, compute the weighted sum (like one neuron). Then do it for 5 examples at once: `batch` has shape (5, 2), `weights` (2,). We want one number per example → shape (5,).

**Expert reasoning:** For one example, `scores @ weights` or `np.dot(scores, weights)` gives a scalar. For a batch, we need matrix × vector: (5, 2) @ (2,) → (5,). NumPy's `@` does exactly that. We keep `weights` as (2,) so the last dimension of `batch` (2) matches; the result loses that dimension → (5,).

In [ ]:
# One example: 2 features, 2 weights
scores = np.array([0.5, 1.5])   # shape (2,)
weights = np.array([2.0, -0.5]) # shape (2,)
one_pred = np.dot(scores, weights)  # or scores @ weights → scalar
print(f"One prediction: {one_pred}, shape {np.array(one_pred).shape}")

# Batch of 5 examples, 2 features each
batch = np.array([[0.5, 1.5], [1.0, 0.0], [0.0, 1.0], [2.0, 2.0], [0.1, 0.9]])  # (5, 2)
batch_pred = batch @ weights  # (5, 2) @ (2,) → (5,)
print(f"Batch predictions: {batch_pred}")
print(f"Shapes: batch {batch.shape}, weights {weights.shape}, result {batch_pred.shape}")

### Your turn [EASY]

Use the same idea: 10 examples, **3** features. `batch_10` has shape (10, 3), `w` has shape (3,). Compute the vector of 10 predictions (shape (10,)).

In [ ]:
np.random.seed(42)
batch_10 = np.random.randn(10, 3)  # 10 examples, 3 features
w = np.array([1.0, -0.5, 0.0])

# YOUR CODE HERE: compute predictions, shape (10,)


### Sensemaking

**What would break if `weights` had shape (3,) and `batch` had shape (5, 2)?** Write one sentence (e.g. dimension mismatch, which dimensions?).

_Your answer: _

---
## Anchor case 2: Normalise a vector to 0–1 [EASY]

**Task:** Given a vector of numbers, rescale so the minimum becomes 0 and the maximum becomes 1. Formula: `(x - min) / (max - min)`.

### Worked example

In [ ]:
v = np.array([10.0, 20.0, 5.0, 35.0, 15.0])
v_min = v.min()
v_max = v.max()
v_normalized = (v - v_min) / (v_max - v_min)
print(f"Original: {v}")
print(f"Normalized: {v_normalized}")
print(f"Min/max of normalized: {v_normalized.min():.2f}, {v_normalized.max():.2f}")

### Your turn [EASY]

Normalise a **random** vector of length 7 (use `np.random.rand(7)`). Check that the result has min 0 and max 1.

In [ ]:
np.random.seed(0)
vec = np.random.rand(7)
# YOUR CODE HERE


### Sensemaking

**When would you use `reshape(-1, 1)` on a 1D vector in an ML context?** (One sentence: what shape are we aiming for and why?)

_Your answer: _

---
## Anchor case 3: Extract elements with a mask (boolean indexing) [MEDIUM]

**Task:** We have predictions (n,) and a boolean mask `valid` (n,) indicating which examples are valid (e.g. no missing label). Extract only the predictions where `valid` is True. Then compute the mean of **only those** predictions.

**Why it matters:** In ML we often filter out invalid or padded positions before computing a loss or metric.

### Worked example

In [ ]:
predictions = np.array([0.1, 0.9, 0.3, 0.7, 0.5])
valid = np.array([True, True, False, True, False])  # 3 valid
pred_valid = predictions[valid]   # boolean indexing → 1D array of length 3
mean_valid = pred_valid.mean()
print(f"predictions: {predictions}")
print(f"valid:       {valid}")
print(f"pred_valid: {pred_valid}")
print(f"mean_valid: {mean_valid}")

### Your turn [MEDIUM]

Given `scores` of shape (20,) and `mask` of shape (20,) (boolean), compute the **maximum** score among the positions where `mask` is True.

In [ ]:
np.random.seed(42)
scores = np.random.rand(20)
mask = np.random.rand(20) > 0.5  # about half True
# YOUR CODE HERE


### Sensemaking

**What is the shape of `predictions[valid]` if `predictions` has shape (100,) and `valid` has 30 True values?**

_Your answer: _

---
## Anchor case 4: View vs copy — avoid a bug [MEDIUM]

**Task:** You take a **slice** of an array (e.g. first 3 elements), modify that slice in-place, and expect the original array to be unchanged. Run the code below: does the original change? Then fix it so the original stays unchanged (use `.copy()` when you need an independent array).

### Worked example: the bug

In [ ]:
original = np.array([10, 20, 30, 40, 50])
slice_view = original[1:4]   # slice → view (shares memory)
slice_view[0] = 999
print(f"After modifying slice_view: original = {original}")
print("The original changed! Slicing returns a VIEW.")

### Your turn [MEDIUM]

Create a **copy** of the first 3 elements, modify the copy (e.g. set copy[0] = -1), and verify that `original` is unchanged.

In [ ]:
original = np.array([10, 20, 30, 40, 50])
# YOUR CODE: get first 3 elements as a COPY, then set copy[0] = -1
first_three = ...  # hint: use .copy()
first_three[0] = -1
print(f"first_three: {first_three}")
print(f"original:    {original}")
assert original[0] == 10, "Original should be unchanged"

### Sensemaking

**When would you prefer a view over a copy?** (Think: memory, speed, safety.)

_Your answer: _

---
## Reference: NumPy essentials (if you need more drill)

The anchor cases above cover the mental model we need for ML. Below is a **subset** of exercises that support those cases (array creation, indexing, shapes). Full set: [NumPy-100](https://github.com/rougier/numpy-100). For each one, think: *"How would I do this with a for loop? Why is the NumPy version faster?"*

### Group A: Basic Array Operations (Exercises 1-10)

**Q1.** Create a null vector of size 10

In [ ]:
# YOUR CODE HERE


**Q2.** Create a null vector of size 10 but the fifth value is 1

In [ ]:
# YOUR CODE HERE


**Q3.** Create a vector with values ranging from 10 to 49

In [ ]:
# YOUR CODE HERE


**Q4.** Reverse a vector (first element becomes last)

In [ ]:
# YOUR CODE HERE


**Q5.** Create a 3x3 matrix with values ranging from 0 to 8

In [ ]:
# YOUR CODE HERE


**Q6.** Find indices of non-zero elements from `[1, 2, 0, 0, 4, 0]`

In [ ]:
# YOUR CODE HERE


**Q7.** Create a 3x3 identity matrix

In [ ]:
# YOUR CODE HERE


### Group B: Indexing and Slicing (Exercises 15-25)

**Q8.** Create a 2D array with 1 on the border and 0 inside (5x5)

In [ ]:
# YOUR CODE HERE


**Q9.** Pad an existing array (3x3 of ones) with a border of zeros

In [ ]:
# YOUR CODE HERE


**Q10.** Create a 5x5 matrix with row values ranging from 0 to 4

In [ ]:
# YOUR CODE HERE


**Q11.** Create a checkerboard 8x8 matrix using slicing

In [ ]:
# YOUR CODE HERE


**Q12.** Normalize a 5x5 random matrix (values between 0 and 1)

In [ ]:
# YOUR CODE HERE


### Group C: Array Manipulation (Exercises 30-40)

**Q13.** How to find common values between two arrays?

In [ ]:
A = np.array([1, 2, 3, 4, 5])
B = np.array([3, 4, 5, 6, 7])

# YOUR CODE HERE


**Q14.** Create a random vector of size 10 and sort it

In [ ]:
# YOUR CODE HERE


**Q15.** Replace the maximum value by 0 in a random vector

In [ ]:
# YOUR CODE HERE


**Q16.** Subtract the mean of each row of a matrix (5x5 random)

In [ ]:
# YOUR CODE HERE


**Q17.** Sort an array by the nth column

In [ ]:
# Create a random 5x3 matrix and sort by column index 1
# YOUR CODE HERE


### Group D: Mathematical Operations (Exercises 45-55)

**Q18.** Compute `((A + B) * (-A / 2))` element-wise without creating intermediate arrays (use `np.add`, `np.multiply`, etc. with `out` parameter)

In [ ]:
A = np.ones(3) * 1
B = np.ones(3) * 2

# YOUR CODE HERE


**Q19.** Get the n largest values of an array

In [ ]:
Z = np.arange(10000)
np.random.shuffle(Z)
n = 5

# YOUR CODE HERE (hint: np.argpartition)


**Q20.** Compute the sum of a vector using different methods and compare speed

In [ ]:
Z = np.random.random(1000000)

# Method 1: Python sum()
%timeit sum(Z)

# Method 2: np.sum()
%timeit np.sum(Z)

# Method 3: Z.sum()
%timeit Z.sum()

# Method 4: np.add.reduce()
%timeit np.add.reduce(Z)

**Q21.** Compute the dot product of two vectors manually (without np.dot), then verify

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Manual dot product
dot_manual = ...

# Verify with NumPy
dot_numpy = np.dot(a, b)

print(f"Manual: {dot_manual}, NumPy: {dot_numpy}, Match: {dot_manual == dot_numpy}")

**Q22.** Compute running (cumulative) sum of a vector

In [ ]:
Z = np.array([1, 2, 3, 4, 5])

# YOUR CODE HERE


### Group E: Linear Algebra Basics (Exercises 60-65)

**Q23.** Multiply a 5x3 matrix by a 3x2 matrix (real matrix product)

In [ ]:
# YOUR CODE HERE


**Q24.** Create a structured array with `name` (string) and `age` (int) fields

In [ ]:
# YOUR CODE HERE


**Q25.** Given a 1D array, negate all elements between 3 and 8 (in-place)

In [ ]:
Z = np.arange(11)

# YOUR CODE HERE


---
## Session 1 Reflection

**What concept was most surprising?**

_Your answer..._

**What still feels fuzzy?**

_Your answer..._

**Connection:** In Session 3 we'll use the same "batch @ weights" idea for linear regression predictions. In Session 5 we assemble everything into one model.